In [1]:
#fill NA with None
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import random
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from statistics import mean
from sklearn import svm
from sklearn.neural_network import MLPClassifier

In [2]:
#https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree
def get_code(tree, feature_names):
        left      = tree.tree_.children_left
        right     = tree.tree_.children_right
        threshold = tree.tree_.threshold
        features  = [feature_names[i] for i in tree.tree_.feature]
        value = tree.tree_.value

        def recurse(left, right, threshold, features, node):
                if (threshold[node] != -2):
                        print("if ( " + features[node] + " <= " + str(threshold[node]) + " ) {")
                        if left[node] != -1:
                                recurse (left, right, threshold, features,left[node])
                        print( "} else {")
                        if right[node] != -1:
                                recurse (left, right, threshold, features,right[node])
                        print( "}")
                else:
                        print( "return " + str(value[node]))

        recurse(left, right, threshold, features, 0)

In [3]:
# data = pd.read_excel('dataset_all_original.xlsx') 
# data = pd.read_excel('dataset_all_gen_ssdp.xlsx') 
# data = pd.read_excel('dataset_all_original_ssdp.xlsx') 
data = pd.read_excel('dataset_all_original_ssdp1_2.xlsx') 
# data = pd.read_excel('dataset_all_original_add_nobias.xlsx') 

In [4]:
data = data.sample(frac = 1) 

In [5]:

data = data.reset_index(drop=True)

In [6]:
data.isnull().sum()

name            0
type            0
vendor          0
DNS           795
mDNS          296
nupnp        4000
open_port     251
ssdp_type     470
ssdp_t        470
ssdp_v        470
dtype: int64

In [7]:
data = data.fillna("NaN")

In [8]:
data.isnull().sum()

name         0
type         0
vendor       0
DNS          0
mDNS         0
nupnp        0
open_port    0
ssdp_type    0
ssdp_t       0
ssdp_v       0
dtype: int64

In [9]:
data.dtypes

name         object
type         object
vendor       object
DNS          object
mDNS         object
nupnp        object
open_port    object
ssdp_type    object
ssdp_t       object
ssdp_v       object
dtype: object

In [10]:
data = data.astype('category')

In [11]:
data.dtypes

name         category
type         category
vendor       category
DNS          category
mDNS         category
nupnp        category
open_port    category
ssdp_type    category
ssdp_t       category
ssdp_v       category
dtype: object

In [12]:
data

,name,type,vendor,DNS,mDNS,nupnp,open_port,ssdp_type,ssdp_t,ssdp_v
0,Foscam,Cameras,Foscam Inc,so-net.ne.jp,_cookley._tcp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,SERVER,Linux/3.14.0 UPnP/1.0 IpBridge/1.33.0
1,Bezeq smarthome,Appliances,Bezeq Group,va.gov,_stim._udp.local.,NaN,tcp/22,NOTIFY * HTTP/1.1,NT,urn:schemas-upnp-org:service:WANIPConnection:1
2,iHome,Appliances,SDI Technologies Inc,acquirethisname.com,_tin._udp.local.,NaN,tcp/22,NOTIFY * HTTP/1.1,LOCATION,http://192.168.0.100:80/description.xml
3,2X Motorola Hubble,Cameras,Hubble Connected,imageshack.us,_namfix._tcp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,SERVER,ipos/7.0 UPnP/1.0 TL-WDR4300/1.0
4,Nest Guard,Appliances,Nest Labs Inc.,reddit.com,_viva._udp.local.,NaN,tcp/443,NOTIFY * HTTP/1.1,NT,urn:dial-multiscreen-org:service:dial:1
5,Dropcam,Cameras,Dropcam,NaN,_it._udp.local.,NaN,NaN,NOTIFY * HTTP/1.1,Host,239.255.255.250:1900
6,Logitech Logi Circle,Cameras,Logitech,squidoo.com,_home ing._tcp.local.,NaN,tcp/22,NOTIFY * HTTP/1.1,Server,SDK 4.3.0.0 UPnP/1.0 MiniUPnPd/1.6
7,Dropcam,Cameras,Dropcam,NaN,NaN,NaN,tcp/1900,NOTIFY * HTTP/1.1,NT,urn:schemas-wifialliance-org:service:WFAWLANCo...
8,HP Printer,Appliances,Hewlett Packard,rediff.com,_zathin._tcp.local.,NaN,tcp/80,NOTIFY * HTTP/1.1,NT,roku:ecp
9,NestCam,Cameras,Nest Labs Inc.,yelp.com,_latlux._udp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,SERVER,"FreeRTOS/6.0.5, UPnP/1.0, IpBridge/0.1"


In [13]:
y = data.drop_duplicates(keep='first')
# y = y.drop(['name'], axis=1)
# y = y.drop(['Name'], axis=1)
y = y.drop_duplicates(keep='first')
# y = y.reset_index()
# shuffle the DataFrame rows
# y = y.sample(frac = 1) 


In [14]:
y = y.astype(str)
len(y)

4000

In [15]:
y = y.apply(LabelEncoder().fit_transform)

In [16]:
list_data = []
list_data.append(y)
len(list_data)

1

In [17]:
accuracy = []
accuracy_t = []
acc = []

In [18]:
for data in list_data:
    for i in range(10):
        accuracy_temp = []
        kf = KFold(n_splits=10, random_state=None, shuffle=True)
#         kf.get_n_splits(data)
        for train_index, test_index in kf.split(data):
        #     print(index)
#             print("TRAIN:", train_index, "TEST:", test_index)
            train, test = data.iloc[train_index], data.iloc[test_index]
#             print("TRAIN:", train, "TEST:", test)
        #     print(len(train))
#             print("TRAIN:", len(train), "TEST:", len(test))
            label_data = train.type
            train_data = train.drop(['type'], axis=1)
            tlabel_data = test.type
            test_data = test.drop(['type'], axis=1)
            truelabel_data = pd.DataFrame(tlabel_data)
            truelabel_data = truelabel_data.reset_index(drop=True)
            clf = MLPClassifier(solver='lbfgs', alpha=1e-5,hidden_layer_sizes=(100,), random_state=1)
            
#             clf = svm.SVC(decision_function_shape='ovo')
            clf = clf.fit(train_data,label_data )
            predict = clf.predict(test_data)
            result_label_data = pd.DataFrame(columns = ['type'])
            result_label_data.type = predict
            compare = pd.DataFrame(columns = ['truelabel_data','predictedlabel'])
            compare.truelabel_data = truelabel_data.type
            compare.predictedlabel = result_label_data.type
#             print(compare)
            t = (accuracy_score(compare.truelabel_data, compare.predictedlabel))
            accuracy_temp.append(t)
            acc.append(t)
            print(i,"  Accuracy: ",t)
        accuracy_t.append(mean(accuracy_temp))
    accuracy.append(mean(accuracy_t))

0   Accuracy:  0.5175
0   Accuracy:  0.4925
0   Accuracy:  0.49
0   Accuracy:  0.4875
0   Accuracy:  0.53
0   Accuracy:  0.515
0   Accuracy:  0.5175
0   Accuracy:  0.4575
0   Accuracy:  0.48
0   Accuracy:  0.5125
1   Accuracy:  0.49
1   Accuracy:  0.5425
1   Accuracy:  0.48
1   Accuracy:  0.475
1   Accuracy:  0.5125
1   Accuracy:  0.51
1   Accuracy:  0.5175
1   Accuracy:  0.5125
1   Accuracy:  0.4525
1   Accuracy:  0.5075
2   Accuracy:  0.57
2   Accuracy:  0.465
2   Accuracy:  0.46
2   Accuracy:  0.4725
2   Accuracy:  0.515
2   Accuracy:  0.4925
2   Accuracy:  0.475
2   Accuracy:  0.5275
2   Accuracy:  0.4925
2   Accuracy:  0.53
3   Accuracy:  0.51
3   Accuracy:  0.465
3   Accuracy:  0.4825
3   Accuracy:  0.5
3   Accuracy:  0.465
3   Accuracy:  0.46
3   Accuracy:  0.545
3   Accuracy:  0.5425
3   Accuracy:  0.4975
3   Accuracy:  0.5325
4   Accuracy:  0.5075
4   Accuracy:  0.5025
4   Accuracy:  0.5075
4   Accuracy:  0.53
4   Accuracy:  0.505
4   Accuracy:  0.4625
4   Accuracy:  0.5025
4 

In [19]:
print(len(y),"\t Accuracy for dataset_gen: \t", accuracy[0])
# print(len(dataset_all_gen_y),"\t Accuracy for dataset_all_gen: \t", accuracy[1])
# print(len(dataset_all_original_y),"\t Accuracy for dataset_all_original: \t", accuracy[2])
# print(len(dataset_jaccard_gen_y),"\t Accuracy for dataset_jaccard_gen: \t", accuracy[3])
# print(len(dataset_jaccard_original_y),"\t Accuracy for dataset_jaccard_original: \t", accuracy[4])
# print(len(dataset_myroom_gen_y),"\t Accuracy for dataset_myroom_gen: \t", accuracy[5])
# print(len(dataset_myroom_original_y),"\t Accuracy for dataset_myroom_original: \t", accuracy[6])
# print(len(dataset_university_gen_y),"\t Accuracy for dataset_university_gen: \t", accuracy[7])
# print(len(dataset_university_original_y),"\t Accuracy for dataset_university_original: \t", accuracy[8])

4000 	 Accuracy for dataset_gen: 	 0.5


In [20]:
print(len(y),"\t STD for dataset_gen: \t", np.std(acc))

4000 	 STD for dataset_gen: 	 0.02435415775591511
